## --Read Bronze Table --

In [0]:
from src.constants import *
from src.common_functions import *
from src.validations import *

import src.common_functions as cf

print(dir(cf))


In [0]:
laborposition_df = spark.table(LABOR_BRONZE_TABLE)

display(laborposition_df)

In [0]:
print(laborposition_df.count())

--Data Validation--

In [0]:
laborposition_df = trim_columns(laborposition_df)

In [0]:
laborposition_df = replace_blank_with_null(laborposition_df)

In [0]:
from pyspark.sql.functions import col

laborposition_df = laborposition_df.withColumn(
    "Labor_Position_Code",
    col("Labor_Position_Code").cast("int")
)

--Remove Duplicates--

In [0]:
laborposition_df = laborposition_df.dropDuplicates(["Labor_Position_Code"])

print("Rows:", laborposition_df.count())

--Null Validation--

In [0]:
from pyspark.sql import functions as F

print("-- Null Validation --")

null_validation_df = laborposition_df.filter(
    F.col("Labor_Position_Code").isNull()
)

print(
    "Rows with NULL Labor_Position_Code:",
    null_validation_df.count()
)

display(null_validation_df)

In [0]:
laborposition_df.printSchema()

In [0]:
display(
    laborposition_df.filter(
        F.col("Labor_Position_Code") == 9510
    )
)

In [0]:
print("Silver Labor Position row count:", laborposition_df.count())

## Add watermark import

In [0]:
from src.watermark import get_watermark, update_watermark
from pyspark.sql import functions as F

print("Watermark framework imported")

## Read the Labor Position watermark

In [0]:
pipeline_name = "Silver Labor Position"
source_name = "Labor_Position.xlsx"

last_watermark = get_watermark(
    pipeline_name,
    source_name
)

print("Last processed watermark:", last_watermark)

## Create the incremental DataFrame

In [0]:
from pyspark.sql import functions as F

if current_labor_watermark is None:

    laborposition_incremental_df = laborposition_df

else:

    laborposition_incremental_df = (
        laborposition_df
        .withColumn(
            "_ingestion_ts",
            F.to_timestamp("ingestion_timestamp")
        )
        .filter(
            F.col("_ingestion_ts") >
            F.lit(current_labor_watermark)
        )
        .drop("_ingestion_ts")
    )

incremental_count = laborposition_incremental_df.count()

print(
    "Incremental Labor Position records:",
    incremental_count
)

In [0]:
current_labor_watermark = get_watermark(
    pipeline_name,
    source_name
)

print(
    "Current Labor Position watermark:",
    current_labor_watermark
)

--Write to silver --

In [0]:
# --Incremental Silver Load--

incremental_count = laborposition_incremental_df.count()

if incremental_count == 0:

    print("No new Labor Position records to load.")
    print("Silver Labor Position table remains unchanged.")

else:

    laborposition_incremental_df.write \
        .mode("append") \
        .saveAsTable(LABOR_POSITION_SILVER_TABLE)

    print(
        f"Incremental Labor Position records loaded: {incremental_count}"
    )

    # Update watermark only after successful Silver load
    new_labor_watermark = (
        laborposition_incremental_df
        .withColumn(
            "_ingestion_ts",
            F.to_timestamp("ingestion_timestamp")
        )
        .agg(
            F.max("_ingestion_ts")
            .alias("max_ingestion_timestamp")
        )
        .collect()[0]["max_ingestion_timestamp"]
    )

    update_watermark(
        pipeline_name,
        source_name,
        new_labor_watermark
    )

    print(
        "Labor Position watermark updated to:",
        new_labor_watermark
    )

--Validate Silver--

In [0]:
silver_df = spark.table(LABOR_SILVER_TABLE)

print(silver_df.count())

display(silver_df)

In [0]:
silver_check_df = spark.table(
    "databricks_project1.silver.labor_position"
)

print("Silver table count:", silver_check_df.count())

display(
    silver_check_df.filter(
        F.col("Labor_Position_Code") == 9510
    )
)

In [0]:
labor_silver_df = spark.table(
    "databricks_project1.silver.labor_position"
)

max_labor_silver_timestamp = (
    labor_silver_df
    .agg(
        F.max("ingestion_timestamp")
        .alias("max_ingestion_timestamp")
    )
    .collect()[0]["max_ingestion_timestamp"]
)

print(
    "Current Silver Labor Position max timestamp:",
    max_labor_silver_timestamp
)

In [0]:
max_labor_source_timestamp = (
    laborposition_df
    .agg(
        F.max("ingestion_timestamp")
        .alias("max_ingestion_timestamp")
    )
    .collect()[0]["max_ingestion_timestamp"]
)

print(
    "Current source Labor Position max timestamp:",
    max_labor_source_timestamp
)

In [0]:
laborposition_df.printSchema()

In [0]:
display(
    laborposition_df
    .filter(
        F.col("Labor_Position_Code") == 9510
    )
)